## 基于 Unsloth 微调DeepSeek-R1-Distill-Llama-8B模型。

本内容为利用Unsloth框架通过LoRA微调DeepSeekR1模型

In [ ]:
# ============================================================
# 固定版本环境安装（CUDA 12.8 + PyTorch 2.10.0）
# ============================================================

# 1. 安装 PyTorch（CUDA 12.8 专用 wheel）
!pip install "torch==2.10.0" torchvision torchaudio \\
    --index-url https://download.pytorch.org/whl/cu128 -q

# 2. 安装 xformers（与 torch 2.10.0 + CUDA 12.8 匹配）
!pip install "xformers==0.0.35" \\
    --index-url https://download.pytorch.org/whl/cu128 -q

# 3. 安装 unsloth 核心（固定版本，避免 API 变动引起冲突）
!pip install "unsloth==2026.3.17" -q
!pip install "unsloth_zoo==2026.3.6" -q

# 4. 安装量化依赖
!pip install "bitsandbytes==0.49.2" -q

# 5. 安装训练框架（版本需与 unsloth 匹配）
!pip install "transformers==4.57.3" -q
!pip install "peft==0.18.1" -q
!pip install "accelerate==1.13.0" -q
!pip install "trl==0.24.0" -q

# 6. 安装数据集库
!pip install "datasets==4.3.0" -q

# 验证关键库版本
import importlib
pkgs = ["torch", "unsloth", "trl", "transformers", "bitsandbytes", "peft", "accelerate", "datasets"]
for pkg in pkgs:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg}: {m.__version__}")
    except Exception as e:
        print(f"{pkg}: 导入失败 -> {e}")

可以使用AutoDL上部署，直接采用框架

## 数据加载

将模型文件、llamafactory解压到autodl-tmp目录下，将数据文件复制到autodl-tmp目录下

## 模型加载

**模型选择方法**
- 选择一个与使用案例相符的模型
- 评估存储、计算能力和数据集
- 选择模型和参数
- 选择基础模型还是指令模型

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 
dtype = None 
load_in_4bit = True # 可以为 False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/root/autodl-tmp/unsloth:DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 数据集准备

In [ ]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

In [ ]:
以下是一条描述任务的指令，配有一个提供更多上下文的输入。请撰写一个恰当完成该请求的回答。在回答之前，请仔细思考问题，并创建一个循序渐进的思维链，以确保回答的逻辑性和准确性。

### 指令：
你是一位医学专家，在临床推理、诊断和治疗规划方面拥有高级知识。请回答以下医学问题。

### 问题：

### 回答：
<think>
{}
</think>
{}"""

用英文会更加准确

In [ ]:
EOS_TOKEN = tokenizer.eos_token  # 一定要添加结束标记，否则会无限生成


def formatting_prompts_func(examples):
    inputs = examples["Question"] # 问题
    cots = examples["Complex_CoT"] # 思考
    outputs = examples["Response"] # 结果
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [ ]:
from datasets import load_dataset
dataset = load_dataset("/root/autodl-tmp/FreedomIntelligence:medical-o1-reasoning-SFT", # 数据集位置
                       'zh', # 语言
                       split = "train[0:500]", #前500条
                       trust_remote_code=True# 信任脚本
                       )

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)

##### text[0] = 
 Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

 Instruction: You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. Please answer the following medical question.

 Question: 患者男性，65岁，突发胸痛，如何诊断？

 Complex CoT: 首先分析症状：胸痛是急性冠脉综合征的典型表现。患者65岁男性，是高危人群。需要立即进行心电图检查。

 Response: 诊断步骤：1.立即心电图；2.查心肌酶；3.排除主动脉夹层；4.必要时冠脉造影。<|end_of_text|>

## 模型训练
使用Huggingface TRL's SFTTrainer

In [ ]:
FastLanguageModel.for_training(model)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

r=16: 标准秩，平衡效果和效率

target_modules: 完整配置，效果最好

lora_alpha=16: 标准缩放，训练稳定

lora_dropout=0: Unsloth 优化，无需 dropout

bias="none": 参数更少，速度更快

use_gradient_checkpointing="unsloth": 显存优化神器

random_state=3407: 确保可复现，Unsloth 测试过的最佳值

use_rslora=False: 标准 LoRA 足够

loftq_config=None: 标准微调不需要

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,#每个设备训练批次大小
        gradient_accumulation_steps = 4,#梯度积累步数
        warmup_steps = 5,#预热步数
        max_steps = 60,#最大训练步数
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",#优化器
        weight_decay = 0.01,
        lr_scheduler_type = "linear",#学习率调度器类型。这里选择 linear 表示学习率将会线性地从初始学习率降低到0。可以帮助模型逐步收敛。
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

**report_to 参数用于指定训练过程中报告的目的地，可以设置为以下几种值：**

"none"：不报告任何信息。这通常用于不需要进行外部监控的本地训练。

"wandb"：将训练日志、模型等信息发送到 Weights & Biases (wandb)，一个用于机器学习实验管理和可视化的平台。如果你希望在训练过程中查看图表、损失函数变化等信息，并跟踪模型的训练过程，选择 wandb。

"tensorboard"：将训练日志发送到 TensorBoard，以便在本地或远程访问并可视化训练过程。可以通过启动 tensorboard 服务器来查看训练日志。

"comet"：将训练日志发送到 Comet，这是一个类似于 Weights & Biases 的机器学习实验跟踪平台。

"mlflow"：将训练日志发送到 MLflow，一个开源的平台，用于管理机器学习生命周期。

"neptune"：将训练日志发送到 Neptune，这是另一个机器学习实验管理平台。

### 微调后调用

In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])